# Random-forest vegetation parent classification for all annual K2 rasters, 2016–2025

This notebook extends the preliminary supervised parent-cluster experiment to every available annual Sentinel-2 K2 harmonic raster from **2016 through 2025**.

The classifier is trained once on the pooled field-supported years:

- 2016
- 2018
- 2025

Response:

- 8 vegetation parent clusters

Predictors:

- IA45 = 15 intercept + 30 amplitude coefficients

Model:

- random forest
- same parameterization as the preliminary notebook

Mask:

`C:\NCA_DATA\templates\keepMask_wc_v200_2021_noCropBuiltWater_buf1.tif`

Outputs:

`C:\NCA_DATA\Analysis\PrelimML_NoSpectralClustering\AllYears_2016_2025`

For each year, the notebook writes:

1. one uint8 vegetation-parent classification raster
2. one 3-band float32 uncertainty raster:
   - maximum class probability
   - probability margin
   - prediction entropy

The model uses a single `predict_proba()` pass per block and derives the hard class from the probability maximum, avoiding a redundant second random-forest inference pass.


In [1]:
from pathlib import Path
import re
import time

import numpy as np
import pandas as pd

import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

from sklearn.ensemble import RandomForestClassifier

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

# =============================================================================
# PATHS
# =============================================================================

TRAINING_CSV = Path(
    r"A:\NCA_DATA\Validation\Supervised_State_Classification"
    r"\FieldVeg_K2_IA45_Training.csv"
)

HARMONIC_ROOT = Path(
    r"A:\NCA_DATA\S2_Harmonics"
)

KEEP_MASK = Path(
    r"C:\NCA_DATA\templates\keepMask_wc_v200_2021_noCropBuiltWater_buf1.tif"
)

OUTPUT_DIR = Path(
    r"C:\NCA_DATA\Analysis\PrelimML_NoSpectralClustering"
    r"\AllYears_2016_2025"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAINING_YEARS = [
    2016,
    2018,
    2025,
]

YEARS_TO_MAP = list(
    range(
        2016,
        2026,
    )
)

EXPECTED_D = 45
CLASS_LABELS = np.arange(
    1,
    9,
    dtype=int,
)

RANDOM_STATE = 42
COMPRESS = "DEFLATE"

print("Training data:", TRAINING_CSV)
print("Harmonic root:", HARMONIC_ROOT)
print("Keep mask:", KEEP_MASK)
print("Output directory:", OUTPUT_DIR)
print("Training years:", TRAINING_YEARS)
print("Years to classify:", YEARS_TO_MAP)


Training data: A:\NCA_DATA\Validation\Supervised_State_Classification\FieldVeg_K2_IA45_Training.csv
Harmonic root: A:\NCA_DATA\S2_Harmonics
Keep mask: C:\NCA_DATA\templates\keepMask_wc_v200_2021_noCropBuiltWater_buf1.tif
Output directory: C:\NCA_DATA\Analysis\PrelimML_NoSpectralClustering\AllYears_2016_2025
Training years: [2016, 2018, 2025]
Years to classify: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


## 1. Load the pooled 2016 / 2018 / 2025 training observations

In [2]:
# =============================================================================
# LOAD TRAINING DATA
# =============================================================================

if not TRAINING_CSV.exists():
    raise FileNotFoundError(
        f"Training CSV not found:\n{TRAINING_CSV}"
    )

df = pd.read_csv(
    TRAINING_CSV,
    low_memory=False,
).drop(
    columns=["Unnamed: 0"],
    errors="ignore",
)

required = {
    "PlotID",
    "Year",
    "parent_cluster",
}

missing = required - set(df.columns)

if missing:
    raise KeyError(
        f"Training data missing columns: {sorted(missing)}"
    )

df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce",
)

df["parent_cluster"] = pd.to_numeric(
    df["parent_cluster"],
    errors="coerce",
)

df = (
    df.loc[
        df["Year"].isin(TRAINING_YEARS)
        & df["parent_cluster"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

df["Year"] = df["Year"].astype(int)
df["parent_cluster"] = df["parent_cluster"].astype(int)
df["PlotID"] = df["PlotID"].astype(str).str.strip()

print("Training rows:", f"{len(df):,}")

display(
    df.groupby(
        ["Year", "parent_cluster"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "n"})
)


Training rows: 479


,Year,parent_cluster,n
0,2016,1,42
1,2016,2,28
2,2016,3,42
3,2016,4,43
4,2016,5,46
5,2016,6,24
6,2016,7,19
7,2016,8,11
8,2018,1,11
9,2018,2,17


## 2. Recover the IA45 predictor columns

In [3]:
# =============================================================================
# IA45
# =============================================================================

intercept_cols = [
    c for c in df.columns
    if re.search(
        r"^(constant_|intercept_)",
        str(c),
        flags=re.IGNORECASE,
    )
]

amplitude_cols = [
    c for c in df.columns
    if re.search(
        r"^(amp|amplitude)",
        str(c),
        flags=re.IGNORECASE,
    )
]

IA_COLS = intercept_cols + amplitude_cols

print("Intercept features:", len(intercept_cols))
print("Amplitude features:", len(amplitude_cols))
print("IA45 dimensions:", len(IA_COLS))

if len(IA_COLS) != EXPECTED_D:
    raise ValueError(
        f"Expected {EXPECTED_D} IA predictors; found {len(IA_COLS)}."
    )

X_df = (
    df[IA_COLS]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

bad = X_df.isna().any(axis=1)

if bad.any():
    display(
        df.loc[
            bad,
            ["PlotID", "Year", "parent_cluster"],
        ].head(50)
    )

    raise ValueError(
        f"{int(bad.sum()):,} rows contain incomplete IA45 predictors."
    )

X = X_df.to_numpy(dtype=np.float32)
y = df["parent_cluster"].to_numpy(dtype=int)

print("X:", X.shape)
print("Observed classes:", sorted(np.unique(y).tolist()))


Intercept features: 15
Amplitude features: 30
IA45 dimensions: 45
X: (479, 45)
Observed classes: [1, 2, 3, 4, 5, 6, 7, 8]


## 3. Fit the pooled random forest

In [4]:
# =============================================================================
# FIT RANDOM FOREST
# =============================================================================

model = RandomForestClassifier(
    n_estimators=800,
    max_features="sqrt",
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    bootstrap=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

t0 = time.perf_counter()

model.fit(
    X,
    y,
)

fit_minutes = (
    time.perf_counter()
    - t0
) / 60.0

print(
    f"Model fit complete in {fit_minutes:.2f} minutes."
)

print(
    "Model classes:",
    model.classes_,
)


Model fit complete in 0.01 minutes.
Model classes: [1 2 3 4 5 6 7 8]


## 4. Confirm all annual K2 rasters and the keep mask

In [5]:
# =============================================================================
# INPUT RASTER REGISTRY
# =============================================================================

def annual_k2_path(
    year,
):
    return (
        HARMONIC_ROOT
        / str(year)
        / f"{year}_K2_InterceptAmpPhase_nobs_RMSE.tif"
    )


missing_rasters = []

for year in YEARS_TO_MAP:

    path = annual_k2_path(year)

    print(
        year,
        "->",
        path,
        "| exists:",
        path.exists(),
    )

    if not path.exists():
        missing_rasters.append(year)


if missing_rasters:
    raise FileNotFoundError(
        "Missing annual K2 raster(s): "
        + ", ".join(map(str, missing_rasters))
    )


if not KEEP_MASK.exists():
    raise FileNotFoundError(
        f"Missing keep mask:\n{KEEP_MASK}"
    )


with rasterio.open(
    KEEP_MASK
) as mask_src:

    print("\nKeep mask")
    print(" shape:", (mask_src.height, mask_src.width))
    print(" CRS:", mask_src.crs)
    print(" transform:", mask_src.transform)
    print(" dtype:", mask_src.dtypes[0])
    print(" nodata:", mask_src.nodata)


2016 -> A:\NCA_DATA\S2_Harmonics\2016\2016_K2_InterceptAmpPhase_nobs_RMSE.tif | exists: True
2017 -> A:\NCA_DATA\S2_Harmonics\2017\2017_K2_InterceptAmpPhase_nobs_RMSE.tif | exists: True
2018 -> A:\NCA_DATA\S2_Harmonics\2018\2018_K2_InterceptAmpPhase_nobs_RMSE.tif | exists: True
2019 -> A:\NCA_DATA\S2_Harmonics\2019\2019_K2_InterceptAmpPhase_nobs_RMSE.tif | exists: True
2020 -> A:\NCA_DATA\S2_Harmonics\2020\2020_K2_InterceptAmpPhase_nobs_RMSE.tif | exists: True
2021 -> A:\NCA_DATA\S2_Harmonics\2021\2021_K2_InterceptAmpPhase_nobs_RMSE.tif | exists: True
2022 -> A:\NCA_DATA\S2_Harmonics\2022\2022_K2_InterceptAmpPhase_nobs_RMSE.tif | exists: True
2023 -> A:\NCA_DATA\S2_Harmonics\2023\2023_K2_InterceptAmpPhase_nobs_RMSE.tif | exists: True
2024 -> A:\NCA_DATA\S2_Harmonics\2024\2024_K2_InterceptAmpPhase_nobs_RMSE.tif | exists: True
2025 -> A:\NCA_DATA\S2_Harmonics\2025\2025_K2_InterceptAmpPhase_nobs_RMSE.tif | exists: True

Keep mask
 shape: (7857, 10435)
 CRS: PROJCS["NAD83 / UTM zone 11N",G

## 5. Match the IA45 names to annual raster band descriptions

In [6]:
# =============================================================================
# BAND MATCHING
# =============================================================================

def get_band_names(
    src,
):
    return [
        (
            str(desc).strip()
            if desc is not None and str(desc).strip()
            else f"band_{i}"
        )
        for i, desc in enumerate(
            src.descriptions,
            start=1,
        )
    ]


def normalize_name(
    x,
):
    return str(x).strip().lower()


def ia_band_indexes(
    src,
    ia_cols,
):

    names = get_band_names(src)

    lookup = {
        normalize_name(name): i
        for i, name in enumerate(
            names,
            start=1,
        )
    }

    indexes = []
    missing = []

    for feature in ia_cols:

        key = normalize_name(feature)

        if key not in lookup:
            missing.append(feature)
        else:
            indexes.append(lookup[key])

    if missing:
        raise KeyError(
            "Missing IA45 raster bands:\n"
            + "\n".join(missing)
        )

    if len(indexes) != EXPECTED_D:
        raise RuntimeError(
            f"Expected {EXPECTED_D} matched raster bands; "
            f"found {len(indexes)}."
        )

    return indexes


for year in YEARS_TO_MAP:

    with rasterio.open(
        annual_k2_path(year)
    ) as src:

        idx = ia_band_indexes(
            src,
            IA_COLS,
        )

        print(
            year,
            "IA45 band count:",
            len(idx),
        )


2016 IA45 band count: 45
2017 IA45 band count: 45
2018 IA45 band count: 45
2019 IA45 band count: 45
2020 IA45 band count: 45
2021 IA45 band count: 45
2022 IA45 band count: 45
2023 IA45 band count: 45
2024 IA45 band count: 45
2025 IA45 band count: 45


## 6. Probability and uncertainty helpers

In [7]:
# =============================================================================
# PROBABILITY HELPERS
# =============================================================================

def align_probabilities(
    proba,
    model_classes,
):

    out = np.zeros(
        (
            len(proba),
            len(CLASS_LABELS),
        ),
        dtype=np.float32,
    )

    lookup = {
        int(cls): j
        for j, cls in enumerate(model_classes)
    }

    for j, cls in enumerate(CLASS_LABELS):

        if int(cls) in lookup:

            out[:, j] = proba[
                :,
                lookup[int(cls)],
            ]

    row_sums = out.sum(
        axis=1,
        keepdims=True,
    )

    valid = row_sums[:, 0] > 0

    out[valid] /= row_sums[valid]

    return out


def predictive_entropy(
    proba,
):

    eps = 1e-12

    return -np.sum(
        proba
        * np.log(
            np.maximum(
                proba,
                eps,
            )
        ),
        axis=1,
    )


def grids_match(
    src_a,
    src_b,
):
    return (
        src_a.width == src_b.width
        and src_a.height == src_b.height
        and src_a.transform == src_b.transform
        and src_a.crs == src_b.crs
    )


## 7. Blockwise all-year classifier

For each valid masked raster block, this notebook reads only IA45, evaluates `predict_proba()` once, derives the hard label by `argmax`, calculates max probability / margin / entropy, and writes the outputs immediately.


In [8]:
# =============================================================================
# BLOCKWISE CLASSIFICATION WITH LIVE ETA
# =============================================================================

def classify_year(
    year,
    model,
):

    src_path = annual_k2_path(year)

    class_path = (
        OUTPUT_DIR
        / f"ParentML_random_forest_{year}_class.tif"
    )

    uncertainty_path = (
        OUTPUT_DIR
        / f"ParentML_random_forest_{year}_uncertainty.tif"
    )

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"CLASSIFYING {year}"
    )

    print(
        "=" * 100
    )

    year_start = time.perf_counter()

    with rasterio.open(
        src_path
    ) as src, rasterio.open(
        KEEP_MASK
    ) as mask_src:

        indexes = ia_band_indexes(
            src,
            IA_COLS,
        )

        use_direct_mask = grids_match(
            src,
            mask_src,
        )

        mask_vrt = None

        if use_direct_mask:

            print(
                "Keep mask matches annual raster grid."
            )

        else:

            print(
                "Keep mask grid differs; using nearest-neighbor WarpedVRT."
            )

            mask_vrt = WarpedVRT(
                mask_src,
                crs=src.crs,
                transform=src.transform,
                width=src.width,
                height=src.height,
                resampling=Resampling.nearest,
            )

        blocks = list(
            src.block_windows(1)
        )

        n_blocks = len(blocks)

        print(
            "Raster shape:",
            f"{src.width:,} × {src.height:,}"
        )

        print(
            "Block shape:",
            src.block_shapes[0]
        )

        print(
            "Number of blocks:",
            f"{n_blocks:,}"
        )

        class_profile = src.profile.copy()

        class_profile.update(
            driver="GTiff",
            count=1,
            dtype="uint8",
            nodata=0,
            compress=COMPRESS,
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        uncertainty_profile = src.profile.copy()

        uncertainty_profile.update(
            driver="GTiff",
            count=3,
            dtype="float32",
            nodata=-9999.0,
            compress=COMPRESS,
            predictor=3,
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        with rasterio.open(
            class_path,
            "w",
            **class_profile,
        ) as class_dst, rasterio.open(
            uncertainty_path,
            "w",
            **uncertainty_profile,
        ) as uncertainty_dst:

            class_dst.set_band_description(
                1,
                "predicted_parent_cluster",
            )

            uncertainty_dst.set_band_description(
                1,
                "max_probability",
            )

            uncertainty_dst.set_band_description(
                2,
                "probability_margin",
            )

            uncertainty_dst.set_band_description(
                3,
                "prediction_entropy",
            )

            total_keep = 0
            total_valid = 0

            for block_number, (
                _,
                window,
            ) in enumerate(
                blocks,
                start=1,
            ):

                if use_direct_mask:

                    keep = mask_src.read(
                        1,
                        window=window,
                        masked=False,
                    )

                else:

                    keep = mask_vrt.read(
                        1,
                        window=window,
                        masked=False,
                    )

                keep_valid = (
                    np.isfinite(keep)
                    & (keep > 0)
                )

                total_keep += int(
                    keep_valid.sum()
                )

                features = src.read(
                    indexes,
                    window=window,
                    masked=False,
                ).astype(
                    np.float32,
                    copy=False,
                )

                features = np.moveaxis(
                    features,
                    0,
                    -1,
                )

                valid = (
                    keep_valid
                    & np.isfinite(
                        features
                    ).all(axis=2)
                )

                if src.nodata is not None:

                    valid &= ~np.any(
                        np.isclose(
                            features,
                            src.nodata,
                        ),
                        axis=2,
                    )

                n_valid = int(
                    valid.sum()
                )

                total_valid += n_valid

                class_block = np.zeros(
                    (
                        int(window.height),
                        int(window.width),
                    ),
                    dtype=np.uint8,
                )

                uncertainty_block = np.full(
                    (
                        3,
                        int(window.height),
                        int(window.width),
                    ),
                    -9999.0,
                    dtype=np.float32,
                )

                if n_valid > 0:

                    X_block = features[valid]

                    raw_proba = model.predict_proba(
                        X_block
                    )

                    proba = align_probabilities(
                        raw_proba,
                        model.classes_,
                    )

                    pred_idx = np.argmax(
                        proba,
                        axis=1,
                    )

                    pred = CLASS_LABELS[
                        pred_idx
                    ].astype(
                        np.uint8
                    )

                    class_block[valid] = pred

                    sorted_p = np.sort(
                        proba,
                        axis=1,
                    )

                    max_p = sorted_p[:, -1]

                    margin = (
                        sorted_p[:, -1]
                        - sorted_p[:, -2]
                    )

                    entropy = predictive_entropy(
                        proba
                    )

                    uncertainty_block[0][valid] = max_p.astype(
                        np.float32
                    )

                    uncertainty_block[1][valid] = margin.astype(
                        np.float32
                    )

                    uncertainty_block[2][valid] = entropy.astype(
                        np.float32
                    )

                class_dst.write(
                    class_block,
                    1,
                    window=window,
                )

                uncertainty_dst.write(
                    uncertainty_block,
                    window=window,
                )

                if (
                    block_number == 1
                    or block_number % 25 == 0
                    or block_number == n_blocks
                ):

                    elapsed = (
                        time.perf_counter()
                        - year_start
                    )

                    rate = (
                        block_number / elapsed
                        if elapsed > 0
                        else np.nan
                    )

                    remaining_blocks = (
                        n_blocks
                        - block_number
                    )

                    eta_seconds = (
                        remaining_blocks / rate
                        if rate > 0
                        else np.nan
                    )

                    print(
                        f"{year} | "
                        f"block {block_number:,}/{n_blocks:,} "
                        f"({100 * block_number / n_blocks:5.1f}%) | "
                        f"elapsed {elapsed / 60:6.1f} min | "
                        f"ETA {eta_seconds / 60:6.1f} min | "
                        f"classified {total_valid:,} px"
                    )

        if mask_vrt is not None:
            mask_vrt.close()

    elapsed_minutes = (
        time.perf_counter()
        - year_start
    ) / 60.0

    print(
        f"\n{year} complete in {elapsed_minutes:.1f} minutes."
    )

    print(
        "Keep-mask pixels:",
        f"{total_keep:,}"
    )

    print(
        "Valid classified pixels:",
        f"{total_valid:,}"
    )

    print(
        "Class raster:",
        class_path
    )

    print(
        "Uncertainty raster:",
        uncertainty_path
    )

    return {
        "Year": year,
        "runtime_minutes": elapsed_minutes,
        "keep_pixels": total_keep,
        "classified_pixels": total_valid,
        "class_path": str(class_path),
        "uncertainty_path": str(uncertainty_path),
    }


## 8. Run all years, 2016–2025

In [9]:
# =============================================================================
# RUN ALL YEARS
# =============================================================================

all_start = time.perf_counter()

results = []

for year in YEARS_TO_MAP:

    result = classify_year(
        year=year,
        model=model,
    )

    results.append(
        result
    )

    pd.DataFrame(
        results
    ).to_csv(
        OUTPUT_DIR
        / "ParentML_random_forest_allYears_progress.csv",
        index=False,
    )


results_df = pd.DataFrame(
    results
)

total_minutes = (
    time.perf_counter()
    - all_start
) / 60.0

print(
    "\n"
    + "=" * 100
)

print(
    "ALL YEARS COMPLETE"
)

print(
    "=" * 100
)

print(
    f"Total runtime: {total_minutes:.1f} minutes "
    f"({total_minutes / 60:.2f} hours)"
)

display(
    results_df
)

results_df.to_csv(
    OUTPUT_DIR
    / "ParentML_random_forest_allYears_outputs.csv",
    index=False,
)



CLASSIFYING 2016
Keep mask matches annual raster grid.
Raster shape: 10,435 × 7,857
Block shape: (512, 512)
Number of blocks: 336
2016 | block 1/336 (  0.3%) | elapsed    0.0 min | ETA    2.2 min | classified 0 px
2016 | block 25/336 (  7.4%) | elapsed    0.2 min | ETA    3.0 min | classified 684,248 px
2016 | block 50/336 ( 14.9%) | elapsed    0.8 min | ETA    4.3 min | classified 3,024,349 px
2016 | block 75/336 ( 22.3%) | elapsed    1.3 min | ETA    4.6 min | classified 5,724,660 px
2016 | block 100/336 ( 29.8%) | elapsed    1.9 min | ETA    4.6 min | classified 8,726,413 px
2016 | block 125/336 ( 37.2%) | elapsed    2.6 min | ETA    4.4 min | classified 12,141,973 px
2016 | block 150/336 ( 44.6%) | elapsed    3.5 min | ETA    4.3 min | classified 16,603,121 px
2016 | block 175/336 ( 52.1%) | elapsed    4.5 min | ETA    4.2 min | classified 21,699,249 px
2016 | block 200/336 ( 59.5%) | elapsed    5.4 min | ETA    3.7 min | classified 26,216,953 px
2016 | block 225/336 ( 67.0%) | el

,Year,runtime_minutes,keep_pixels,classified_pixels,class_path,uncertainty_path
0,2016,7.628090,36619399,36367546,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...
1,2017,7.324889,36619399,36477798,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...
2,2018,7.697494,36619399,36493754,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...
3,2019,8.177671,36619399,36515630,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...
4,2020,8.234008,36619399,36545680,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...
5,2021,7.886826,36619399,36541682,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...
6,2022,7.775157,36619399,36538126,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...
7,2023,7.764995,36619399,36549241,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...
8,2024,7.873735,36619399,36543369,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...
9,2025,7.830823,36619399,36527562,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...,C:\NCA_DATA\Analysis\PrelimML_NoSpectralCluste...


## 9. Class-frequency summaries for every annual map

In [10]:
# =============================================================================
# ANNUAL CLASS FREQUENCIES
# =============================================================================

annual_summaries = []

for year in YEARS_TO_MAP:

    class_path = (
        OUTPUT_DIR
        / f"ParentML_random_forest_{year}_class.tif"
    )

    with rasterio.open(
        class_path
    ) as src:

        arr = src.read(
            1,
            masked=False,
        )

    valid = (
        arr > 0
    )

    n_valid = int(
        valid.sum()
    )

    values, counts = np.unique(
        arr[valid],
        return_counts=True,
    )

    summary = pd.DataFrame({
        "Year": year,
        "parent_cluster": values.astype(int),
        "n_pixels": counts.astype(np.int64),
    })

    summary["percent"] = (
        100.0
        * summary["n_pixels"]
        / n_valid
    )

    annual_summaries.append(
        summary
    )

    print(
        "\n",
        year
    )

    display(
        summary
    )


annual_summary_df = pd.concat(
    annual_summaries,
    ignore_index=True,
)

annual_summary_df.to_csv(
    OUTPUT_DIR
    / "ParentML_random_forest_allYears_class_frequencies.csv",
    index=False,
)



 2016


,Year,parent_cluster,n_pixels,percent
0,2016,1,6129083,16.853166
1,2016,2,2359137,6.486929
2,2016,3,8528767,23.451588
3,2016,4,6732661,18.512827
4,2016,5,6438887,17.705036
5,2016,6,3381215,9.297342
6,2016,7,1514629,4.164782
7,2016,8,1283167,3.528330



 2017


,Year,parent_cluster,n_pixels,percent
0,2017,1,7034573,19.284533
1,2017,2,1876118,5.143178
2,2017,3,8134183,22.298997
3,2017,4,7111129,19.494403
4,2017,5,4320845,11.845137
5,2017,6,6257404,17.154007
6,2017,7,929630,2.548482
7,2017,8,813916,2.231264



 2018


...

,Year,parent_cluster,n_pixels,percent
0,2023,1,6892510,18.858148
1,2023,2,3154976,8.632125
2,2023,3,8563991,23.431379
3,2023,4,11310471,30.945844
4,2023,5,1613476,4.414527
5,2023,6,1696967,4.642961
6,2023,7,2370659,6.486206
7,2023,8,946191,2.588812



 2024


,Year,parent_cluster,n_pixels,percent
0,2024,1,1651283,4.518694
1,2024,2,780220,2.135052
2,2024,3,4307109,11.786294
3,2024,4,20997655,57.459549
4,2024,5,3781740,10.348635
5,2024,6,909491,2.488799
6,2024,7,2714401,7.427889
7,2024,8,1401470,3.835087



 2025


,Year,parent_cluster,n_pixels,percent
0,2025,1,7134288,19.531246
1,2025,2,2660132,7.282534
2,2025,3,2825324,7.734773
3,2025,4,15109794,41.365460
4,2025,5,702511,1.923235
5,2025,6,3774954,10.334536
6,2025,7,2250347,6.160682
7,2025,8,2070212,5.667534


## 10. Uncertainty summaries

In [11]:
# =============================================================================
# ANNUAL UNCERTAINTY SUMMARIES
# =============================================================================

uncertainty_rows = []

band_names = {
    1: "max_probability",
    2: "probability_margin",
    3: "prediction_entropy",
}

for year in YEARS_TO_MAP:

    path = (
        OUTPUT_DIR
        / f"ParentML_random_forest_{year}_uncertainty.tif"
    )

    with rasterio.open(
        path
    ) as src:

        for band_no, name in band_names.items():

            arr = src.read(
                band_no,
                masked=True,
            )

            values = arr.compressed()

            uncertainty_rows.append({
                "Year": year,
                "metric": name,
                "n": len(values),
                "mean": float(np.mean(values)),
                "sd": float(np.std(values)),
                "p05": float(np.quantile(values, 0.05)),
                "p25": float(np.quantile(values, 0.25)),
                "median": float(np.median(values)),
                "p75": float(np.quantile(values, 0.75)),
                "p95": float(np.quantile(values, 0.95)),
            })


uncertainty_summary = pd.DataFrame(
    uncertainty_rows
)

display(
    uncertainty_summary.round(4)
)

uncertainty_summary.to_csv(
    OUTPUT_DIR
    / "ParentML_random_forest_allYears_uncertainty_summary.csv",
    index=False,
)


,Year,metric,n,mean,sd,p05,p25,median,p75,p95
0,2016,max_probability,36367546,0.3438,0.0980,0.2099,0.2707,0.3291,0.4022,0.5267
1,2016,probability_margin,36367546,0.1311,0.1150,0.0075,0.0416,0.0989,0.1908,0.3610
2,2016,prediction_entropy,36367546,1.7343,0.1777,1.4079,1.6243,1.7547,1.8687,1.9865
3,2017,max_probability,36477798,0.3343,0.0931,0.2106,0.2662,0.3192,0.3855,0.5153
4,2017,probability_margin,36477798,0.1306,0.1145,0.0076,0.0420,0.0983,0.1877,0.3683
5,2017,prediction_entropy,36477798,1.7655,0.1561,1.4881,1.6655,1.7856,1.8861,1.9801
6,2018,max_probability,36493754,0.3751,0.1221,0.2175,0.2815,0.3505,0.4470,0.6194
7,2018,probability_margin,36493754,0.1701,0.1490,0.0091,0.0517,0.1259,0.2498,0.4855
8,2018,prediction_entropy,36493754,1.6875,0.2128,1.2594,1.5687,1.7276,1.8466,1.9642
9,2019,max_probability,36515630,0.3379,0.0939,0.2103,0.2672,0.3225,0.3950,0.5168


## Interpretation

These annual maps are preliminary supervised products, not independently discovered ecological states.

The current label space contains only the eight vegetation parents, so bare ground, dark surfaces, SATR litter, and other explicit spectral endmembers are still forced into vegetation classes. These outputs therefore provide the pre-endmember baseline for later comparison.
